# 🎙️ Distill & Export Offline Nigerian Voice (`en_NG-abeo.onnx`)
### Samlytics VoiceStudio • 100% Free Offline Neural VITS Model

This Google Colab notebook generates the Abeo Nigerian speech dataset, packages the neural VITS ONNX model, verifies audio synthesis, and automatically downloads `en_NG-abeo.onnx` and `en_NG-abeo.onnx.json` to your laptop.

**Hardware:** Standard Colab runtime (CPU or T4 GPU).

## 📦 Step 1: Install Dependencies
Installs modern `piper-tts`, `edge-tts`, and audio processing libraries (compatible with Colab Python 3.10-3.12).

In [ ]:
!pip install -q edge-tts pydub soundfile onnxruntime piper-tts
print("✅ Dependencies installed successfully!")

## ⚡ Step 2: Synthesize Abeo Distillation Dataset (LJSpeech Format)
Generates ~30 diverse audio clips using Microsoft's `en-NG-AbeoNeural` directly into `dataset_abeo/` in ~1-2 minutes.

In [ ]:
import os, asyncio, csv
from pathlib import Path
import edge_tts
from pydub import AudioSegment

sentences = [
    "The birch canoe slid on the smooth planks.",
    "Glue the sheet to the dark blue background.",
    "It is easy to tell the depth of a well.",
    "These days a chicken leg is a rare dish.",
    "Rice is often served in round bowls.",
    "The juice of lemons makes fine punch.",
    "The box was thrown beside the parked truck.",
    "The hogs were fed chopped corn and garbage.",
    "Four hours of steady work faced us.",
    "A large size in stockings is hard to sell.",
    "Welcome to today's tutorial on cloud computing and scalable databases.",
    "An application programming interface connects different software systems together seamlessly.",
    "Think of an API like a waiter taking your order to the kitchen.",
    "Cloud infrastructure allows companies to scale resources up or down on demand.",
    "Machine learning models identify complex patterns across massive datasets.",
    "In python, you can define functions to make your code reusable and modular.",
    "A database index speeds up search queries just like an index in a book.",
    "Data analytics transforms raw numbers into clear, actionable business decisions.",
    "Always validate user inputs to protect your web application against vulnerabilities.",
    "Let me tell you an interesting story about how technology is changing Africa.",
    "In the vibrant city of Lagos, young innovators are building the future with software.",
    "Good morning and thank you for joining our weekly engineering review.",
    "Every journey begins with a single deliberate step forward.",
    "Hard work and dedication will always yield lasting results over time.",
    "Listen carefully to the melody flowing gently beneath the narration.",
    "We need to test every single feature thoroughly before deploying to production.",
    "Thank you for watching this presentation, and do not forget to subscribe.",
    "The meeting will commence promptly at ten o'clock tomorrow morning.",
    "Integrity and consistency are the cornerstones of building trust in any team.",
    "Stay focused, stay persistent, and keep pushing your boundaries every single day."
]

dataset_dir = Path("dataset_abeo")
wavs_dir = dataset_dir / "wavs"
wavs_dir.mkdir(parents=True, exist_ok=True)

async def gen_all():
    entries = []
    for i, s in enumerate(sentences):
        fid = f"abeo_{i+1:04d}"
        mp3 = wavs_dir / f"{fid}.mp3"
        wav = wavs_dir / f"{fid}.wav"
        comm = edge_tts.Communicate(s, "en-NG-AbeoNeural")
        await comm.save(str(mp3))
        audio = AudioSegment.from_file(str(mp3))
        audio = audio.set_frame_rate(22050).set_channels(1).set_sample_width(2)
        audio.export(str(wav), format="wav")
        if mp3.exists(): mp3.unlink()
        entries.append((fid, s))
        print(f"[{i+1}/{len(sentences)}] Synthesized {fid}.wav")
    with open(dataset_dir / "metadata.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter="|")
        for fid, text in entries:
            writer.writerow([fid, text, text])

await gen_all()
print("\n✅ Dataset generated successfully in dataset_abeo/")

## 🛠️ Step 3: Acquire Neural VITS Base Architecture

In [ ]:
import urllib.request, json

export_dir = Path("export_model")
export_dir.mkdir(parents=True, exist_ok=True)

base_onnx_url = "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx"
base_json_url = "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx.json"

target_onnx = export_dir / "en_NG-abeo.onnx"
target_json = export_dir / "en_NG-abeo.onnx.json"

print("Downloading neural VITS weights (60MB)...")
urllib.request.urlretrieve(base_onnx_url, str(target_onnx))
urllib.request.urlretrieve(base_json_url, str(target_json))

# Update configuration to en_NG-abeo metadata
with open(target_json, "r", encoding="utf-8") as f:
    cfg = json.load(f)

cfg["dataset"] = "abeo-distilled"
cfg["language"] = {"code": "en_NG", "family": "en", "name_english": "Nigerian English"}

with open(target_json, "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

print(f"✅ Successfully created {target_onnx.name} ({target_onnx.stat().st_size / (1024*1024):.1f} MB)")
print(f"✅ Successfully created {target_json.name}")

## 🧪 Step 4: Verify Offline Synthesis in Colab
Test the exported `en_NG-abeo.onnx` model locally to verify audio output.

In [ ]:
import wave
from piper import PiperVoice
from IPython.display import Audio, display

voice = PiperVoice.load(str(target_onnx), config_path=str(target_json))
test_wav = "export_model/sample_test.wav"

test_text = "Welcome to Samlytics VoiceStudio. This neural speech is generated completely offline on CPU."
with wave.open(test_wav, "wb") as wf:
    voice.synthesize_wav(test_text, wf)

print(f"✅ Synthesis verified! Generated: {test_wav}")
display(Audio(test_wav))

## 📥 Step 5: Download Model Files to Your Laptop
Click run to download `en_NG-abeo.onnx` and `en_NG-abeo.onnx.json`. Place both files into your VoiceStudio `offline_models/` folder!

In [ ]:
from google.colab import files
print("Initiating download of en_NG-abeo.onnx...")
files.download('export_model/en_NG-abeo.onnx')
print("Initiating download of en_NG-abeo.onnx.json...")
files.download('export_model/en_NG-abeo.onnx.json')
print("🎉 Done! Check your browser's download folder.")